In [9]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


In [10]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Path to dataset files: /kaggle/input/creditcardfraud


In [11]:
import os

# The `path` variable from the previous cell holds the directory path: '/kaggle/input/creditcardfraud'
# Assuming the CSV file is named 'creditcard.csv' inside this directory
csv_file_path = os.path.join(path, "creditcard.csv")

df = pd.read_csv(csv_file_path)
print("Dataset shape:", df.shape)
print("\nFirst five rows:")
print(df.head())

Dataset shape: (284807, 31)

First five rows:
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 

In [12]:
print(df.isnull().sum())
print(df.info())

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11  

In [14]:
df_small, _ = train_test_split(
    df,
    train_size=20000,
    stratify=df["Class"],
    random_state=42
)

print("Original dataset:", df.shape)
print("Small dataset:", df_small.shape)


x = df_small.drop("Class", axis=1)
y = df_small["Class"]

Original dataset: (284807, 31)
Small dataset: (20000, 31)


In [15]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
smote = SMOTE(random_state=42)

x_train_smote, y_train_smote = smote.fit_resample(
    x_train,
    y_train
)

print("Normal cases:", np.sum(y_train_smote == 0))
print("Fraud cases:", np.sum(y_train_smote == 1))


Normal cases: 15972
Fraud cases: 15972


In [17]:
model = XGBClassifier(
    n_estimators=10,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model.fit(x_train_smote, y_train_smote)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=10, n_jobs=None,
              num_parallel_tree=None, ...)

In [18]:
y_prob = model.predict_proba(x_test)[:, 1]

y_pred = (y_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\nXGBoost Results - Threshold 0.5")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



XGBoost Results - Threshold 0.5
Accuracy: 0.999
Precision: 0.6666666666666666
Recall: 0.8571428571428571
F1: 0.75

Confusion Matrix:
[[3990    3]
 [   1    6]]


In [19]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]

print("\nThreshold Tuning")

for threshold in thresholds:

    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )

    print(
        "Threshold:", threshold,
        "Precision:", precision,
        "Recall:", recall,
        "F1:", f1
    )


Threshold Tuning
Threshold: 0.2 Precision: 0.0425531914893617 Recall: 0.8571428571428571 F1: 0.08108108108108109
Threshold: 0.3 Precision: 0.24 Recall: 0.8571428571428571 F1: 0.375
Threshold: 0.4 Precision: 0.4 Recall: 0.8571428571428571 F1: 0.5454545454545454
Threshold: 0.5 Precision: 0.6666666666666666 Recall: 0.8571428571428571 F1: 0.75
Threshold: 0.6 Precision: 1.0 Recall: 0.8571428571428571 F1: 0.9230769230769231


In [20]:
feature_importance = pd.DataFrame({
    "Feature": x.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance:")
print(feature_importance)


Feature Importance:
   Feature  Importance
10     V10    0.732528
14     V14    0.080989
26     V26    0.040067
4       V4    0.032052
21     V21    0.027189
11     V11    0.018738
7       V7    0.016744
9       V9    0.011043
12     V12    0.009150
16     V16    0.006440
17     V17    0.005395
5       V5    0.004421
19     V19    0.002775
18     V18    0.002716
3       V3    0.002684
8       V8    0.002221
0     Time    0.002104
2       V2    0.001029
24     V24    0.001013
28     V28    0.000330
6       V6    0.000195
22     V22    0.000176
13     V13    0.000000
1       V1    0.000000
20     V20    0.000000
15     V15    0.000000
25     V25    0.000000
23     V23    0.000000
27     V27    0.000000
29  Amount    0.000000


In [21]:
svm = SVC(
    kernel="linear",
    C=1,
    class_weight="balanced"
)

svm.fit(x_train, y_train)

SVC(C=1, class_weight='balanced', kernel='linear')

In [22]:
y_pred_svm = svm.predict(x_test)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print("\nSVM Results")
print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1:", f1_svm)

print("\nSVM Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))


SVM Results
Accuracy: 0.98725
Precision: 0.1206896551724138
Recall: 1.0
F1: 0.2153846153846154

SVM Confusion Matrix:
[[3942   51]
 [   0    7]]


In [23]:
print("\n========== MODEL COMPARISON ==========")

print("\nXGBoost")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print("\nSVM")
print("Accuracy:", accuracy_svm)
print("Precision:", precision_svm)
print("Recall:", recall_svm)
print("F1:", f1_svm)


========== MODEL COMPARISON ==========

XGBoost
Accuracy: 0.999
Precision: 0.6666666666666666
Recall: 0.8571428571428571
F1: 0.75

SVM
Accuracy: 0.98725
Precision: 0.1206896551724138
Recall: 1.0
F1: 0.2153846153846154
